In [ ]:
import json
import os
import random

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50
from tqdm.auto import tqdm
from sklearn.metrics import f1_score  # criterio de best-checkpoint (F1-macro en validación)
import wandb  # pip install wandb, si no esta ya en el entorno del workstation

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

In [ ]:
SEED = 42


def seed_everything(seed=SEED):
    """Fija todas las fuentes de aleatoriedad del notebook.

    Cubre la init de la cabeza (linear1/bn1/linear2 en FullModel), el
    shuffle del DataLoader de train y la RandomRotation/RandomHorizontalFlip
    del train_transform (usan el RNG global de torch porque el DataLoader
    corre con num_workers=0). Mismo patrón que scripts/gen_lr_backbone_grid.py
    (exp10-exp19) — mantenerlo consistente si se copia esta celda a nuevos
    notebooks del bloque exp24+.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinismo en cuDNN: reproducible a costa de algo de velocidad.
    # Con benchmark=True cuDNN elige algoritmos según el hardware y la
    # misma semilla deja de dar el mismo resultado entre máquinas.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything()
print(f"Semilla fijada: {SEED} (cudnn.deterministic=True, benchmark=False)")

In [ ]:
EXP_ID = "exp27"
RUN_NAME = "exp27_resnet50_radimagenet_mammobench_all_unfrozen_mlp03_ce_lr1e-4_img512_bs16_wandb"
UNFROZEN_DESC = "todo el backbone descongelado"
BLOCK = 6  # bloque 6: profundidad de descongelamiento (exp23-exp27), sobre la base con seed+wandb+CE+F1-macro
# Índices de backbone (nn.Sequential) a descongelar — ver freeze cell para el mapa completo.
UNFREEZE_IDX = [7, 6, 5, 4, 1, 0]
HEAD = "mlp"

LR = 1e-4            # cabeza (linear1/bn1/linear2)
LR_BACKBONE = 1e-4   # backbone descongelado — igual a LR, sin LR diferencial (intencional)
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 16
NUM_EPOCHS = 100
PATIENCE = 20
IMAGE_SIZE = 512
ROTATION_DEGREES = 7

PROJECT_ROOT = Path(
    "/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/13-PregradoJulian/"
    "Federal Learning/infraestructura federada/Federal-Learning"
)

DATASET_ROOT = Path(
    "/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/02-Databases/Mammo-Bench/"
    "c86fb00c-0fb8-4e0e-85a2-4d415f9c1ada_1a9410d8-9769-4064-a064-0160f2fd193d_"
    "DATASET-FILE_Mammo_Bench_zip_20241225112148174/Mammo_Data/Mammo-Bench"
)
MANIFEST_PATH = PROJECT_ROOT / "manifests" / "fedmammobench_tompei.csv"
RADIMAGENET_WEIGHTS_PATH = PROJECT_ROOT / "weights" / "RadImageNet-resnet50.pth"

# W&B: la API key NUNCA se pega aquí. Se lee de la variable de entorno
# WANDB_API_KEY, o de las credenciales cacheadas en ~/.netrc tras correr
# `wandb login` una vez en este workstation (a diferencia de los
# contenedores Docker del paquete, aqui el notebook SI hereda ese
# ~/.netrc porque corre directo en el host, sin aislar el filesystem).
WANDB_PROJECT = "fedmammobench"
WANDB_ENTITY = None  # None -> entity por defecto de la API key
WANDB_MODE = "online"  # "online" | "offline" | "disabled"
WANDB_TAGS = ["notebook", "centralized", "radimagenet", "mammobench", "all_unfrozen"]

RUN_DIR = PROJECT_ROOT / "runs" / RUN_NAME
PLOTS_DIR = RUN_DIR / "plots"
RUN_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = RUN_DIR / "best_model.pth"
LAST_MODEL_PATH = RUN_DIR / "last_model.pth"

print(f"RUN_DIR: {RUN_DIR}")

In [ ]:
# W&B init. `WandbWriter` (src/fedmammobench/utils/wandb_utils.py) hace este
# mismo degradado en el paquete; aqui se replica a mano porque el notebook no
# importa fedmammobench. Nunca debe bloquear ni pedir la key por input():
# si no hay credenciales, cae a modo offline (o se desactiva del todo).
os.environ.setdefault("WANDB_SILENT", "true")

wandb_run = None
if WANDB_MODE != "disabled":
    mode = WANDB_MODE
    has_creds = bool(os.environ.get("WANDB_API_KEY", "").strip()) or bool(wandb.api.api_key)
    if mode == "online" and not has_creds:
        print(
            "No hay WANDB_API_KEY ni credenciales cacheadas (~/.netrc); "
            "corriendo wandb en modo offline. Sincroniza despues con "
            "`wandb sync` sobre la carpeta wandb/ de este run."
        )
        mode = "offline"

    try:
        wandb_run = wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name=RUN_NAME,
            group=EXP_ID,
            tags=WANDB_TAGS,
            mode=mode,
            dir=str(RUN_DIR),
            config={
                "exp_id": EXP_ID,
                "seed": SEED,
                "block": BLOCK,
                "unfrozen": UNFROZEN_DESC,
                "unfreeze_idx": UNFREEZE_IDX,
                "head": HEAD,
                "loss": "CrossEntropyLoss",
                "lr": LR,
                "lr_backbone": LR_BACKBONE if HEAD == "mlp" else None,
                "dropout": DROPOUT if HEAD == "mlp" else None,
                "weight_decay": WEIGHT_DECAY,
                "batch_size": BATCH_SIZE,
                "num_epochs": NUM_EPOCHS,
                "patience": PATIENCE,
                "image_size": IMAGE_SIZE,
                "rotation_degrees": ROTATION_DEGREES,
            },
        )
        print(f"W&B run ({mode}): {getattr(wandb_run, 'url', None) or RUN_DIR / 'wandb'}")
    except Exception as exc:
        print(f"wandb.init() fallo ({exc!r}); se continua sin W&B.")
        wandb_run = None
else:
    print("W&B desactivado (WANDB_MODE = 'disabled').")

In [ ]:
SPLIT_COL = "split"
LABEL_COL = "classification"
PATH_COL = "preprocessed_image_path"

TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT = "train", "val", "test"


class CSVDataset(Dataset):
    def __init__(self, csv_file, split, root_dir="", transform=None, label_to_idx=None):
        self.data = pd.read_csv(csv_file)

        missing = {SPLIT_COL, LABEL_COL, PATH_COL} - set(self.data.columns)
        if missing:
            raise KeyError(
                f"El manifest {csv_file} no tiene la(s) columna(s) {sorted(missing)}. "
                f"Columnas disponibles: {list(self.data.columns)}"
            )

        available = sorted(self.data[SPLIT_COL].dropna().unique())
        if split not in available:
            raise ValueError(
                f"split={split!r} no existe en la columna {SPLIT_COL!r}. "
                f"Valores disponibles: {available}"
            )

        self.data = self.data[self.data[SPLIT_COL] == split].reset_index(drop=True)
        if len(self.data) == 0:
            raise ValueError(f"El split {split!r} quedó vacío tras filtrar {csv_file}.")

        self.root_dir = Path(root_dir)
        self.transform = transform

        if label_to_idx is None:
            labels = sorted(self.data[LABEL_COL].unique())
            self.label_to_idx = {label: i for i, label in enumerate(labels)}
        else:
            self.label_to_idx = label_to_idx

        unknown = set(self.data[LABEL_COL].unique()) - set(self.label_to_idx)
        if unknown:
            raise ValueError(
                f"El split {split!r} tiene etiquetas {sorted(unknown)} ausentes en "
                f"label_to_idx={self.label_to_idx}."
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        img_path = Path(self.root_dir) / Path(row[PATH_COL])
        image = Image.open(img_path).convert("RGB")

        label = self.label_to_idx[row[LABEL_COL]]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
_manifest = pd.read_csv(MANIFEST_PATH)

_required = {SPLIT_COL, LABEL_COL, PATH_COL}
assert _required <= set(_manifest.columns), (
    f"Faltan columnas en el manifest: {sorted(_required - set(_manifest.columns))}"
)

_splits = sorted(_manifest[SPLIT_COL].dropna().unique())
assert set(_splits) >= {TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT}, (
    f"El manifest solo tiene splits {_splits}; se esperaban "
    f"{[TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT]}"
)

print(f"Filas totales: {len(_manifest)}\n")
print(pd.crosstab(_manifest[SPLIT_COL], _manifest[LABEL_COL], margins=True))

if "source_dataset" in _manifest.columns:
    print()
    print(pd.crosstab(_manifest["source_dataset"], _manifest[SPLIT_COL], margins=True))

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(ROTATION_DEGREES),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [ ]:
train_processed = CSVDataset(
    csv_file=MANIFEST_PATH, split=TRAIN_SPLIT, root_dir=DATASET_ROOT,
    transform=train_transform,
)
validation_processed = CSVDataset(
    csv_file=MANIFEST_PATH, split=VAL_SPLIT, root_dir=DATASET_ROOT,
    transform=val_transform, label_to_idx=train_processed.label_to_idx,
)
test_processed = CSVDataset(
    csv_file=MANIFEST_PATH, split=TEST_SPLIT, root_dir=DATASET_ROOT,
    transform=val_transform, label_to_idx=train_processed.label_to_idx,
)

print(f"label_to_idx: {train_processed.label_to_idx}")
print(f"train: {len(train_processed)} | val: {len(validation_processed)} | test: {len(test_processed)}")

In [ ]:
# drop_last=True: la cabeza usa BatchNorm1d, que revienta si el último batch
# de la época queda con 1 sola muestra (7057 % 16 == 1 con BATCH_SIZE=16).
#
# El shuffle de este DataLoader consume el RNG global de torch; con un
# generator propio sembrado el orden de los batches es el mismo en cada
# ejecución aunque se re-ejecuten celdas de arriba en otro orden.
_loader_generator = torch.Generator()
_loader_generator.manual_seed(SEED)


def _seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


train = DataLoader(
    train_processed, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    generator=_loader_generator, worker_init_fn=_seed_worker,
)
validation = DataLoader(validation_processed, batch_size=BATCH_SIZE, shuffle=False)
test = DataLoader(test_processed, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class Classifier(nn.Module):
    def __init__(self, num_class):
        super().__init__()
        self.linear = nn.Linear(2048, num_class)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.linear(x)


class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        base_model = resnet50(weights=None)
        encoder_layers = list(base_model.children())
        self.backbone = nn.Sequential(*encoder_layers[:9])

    def forward(self, x):
        return self.backbone(x)

In [ ]:
def load_radimagenet_backbone(weights_path, device="cpu"):
    model = resnet50(weights=None)
    checkpoint = torch.load(weights_path, map_location=device)

    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

    # RadImageNet-resnet50.pth guarda el backbone como nn.Sequential, así que
    # sus claves son "backbone.0.weight" en vez de "conv1.weight". Sin este
    # remap, load_state_dict(strict=False) casa CERO claves y deja el modelo
    # en su init aleatorio sin avisar.
    backbone_remap = {
        "backbone.0.": "conv1.",
        "backbone.1.": "bn1.",
        "backbone.4.": "layer1.",
        "backbone.5.": "layer2.",
        "backbone.6.": "layer3.",
        "backbone.7.": "layer4.",
    }
    remapped_state_dict = {}
    for k, v in state_dict.items():
        new_k = k
        for old_prefix, new_prefix in backbone_remap.items():
            if k.startswith(old_prefix):
                new_k = new_prefix + k[len(old_prefix):]
                break
        remapped_state_dict[new_k] = v
    state_dict = remapped_state_dict

    state_dict = {
        k: v for k, v in state_dict.items()
        if k.startswith(("conv1", "bn1", "relu", "maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool"))
    }

    if len(state_dict) == 0:
        raise RuntimeError(
            "0 RadImageNet tensors matched the model after remapping — "
            "check the checkpoint's key format before continuing."
        )

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded {len(state_dict)} RadImageNet tensors into the backbone.")
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)

    # Trunca a conv1..avgpool para que la salida (batch, 2048, 1, 1) encaje
    # con el Linear(2048, ...) del clasificador.
    encoder_layers = list(model.children())
    return nn.Sequential(*encoder_layers[:9])


backbone = load_radimagenet_backbone(RADIMAGENET_WEIGHTS_PATH, device=device).to(device)

In [ ]:
# Índices de backbone (nn.Sequential, ver backbone_remap en load_radimagenet_backbone):
# 0=conv1, 1=bn1, 4=layer1, 5=layer2, 6=layer3, 7=layer4 (2=relu/3=maxpool/8=avgpool
# no tienen parámetros). UNFREEZE_IDX se fija en la celda de parámetros — esta celda
# es la misma para las 5 profundidades del bloque, solo cambia UNFREEZE_IDX.
for param in backbone.parameters():
    param.requires_grad = False
for idx in UNFREEZE_IDX:
    for param in backbone[idx].parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total = sum(p.numel() for p in backbone.parameters())
pct = 100 * trainable / total
print(f"[{EXP_ID}] Backbone trainable params: {trainable:,} / {total:,} ({pct:.2f}%)")

In [ ]:
train_loss_history = []
val_loss_history = []
val_f1_history = []


def freeze_bn_running_stats(module):
    # requires_grad=False congela gamma/beta pero NO running_mean/running_var,
    # que se siguen actualizando en cada forward en modo train(). eval() las fija.
    for m in module.modules():
        if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
            m.eval()


def train_model(model, criterion, optimizer, num_epochs=NUM_EPOCHS, sheduler=None,
                patience=PATIENCE):
    # Best-checkpoint y early stopping se rigen por F1-macro en validación, no
    # por la loss: con clases desbalanceadas la loss puede seguir bajando
    # (el modelo se vuelve más "seguro") mientras el F1 de la clase minoritaria
    # empeora, así que es un mejor criterio de "mejor modelo" que la loss sola.
    best_val_f1 = -1.0
    train_loss_history.clear()
    val_loss_history.clear()
    val_f1_history.clear()
    counter = 0
    best_epoch = 0

    epoch_bar = tqdm(range(num_epochs), desc="Epochs", unit="epoch")
    for e in epoch_bar:
        train_loss = 0.0
        model.train()
        freeze_bn_running_stats(model)
        train_bar = tqdm(train, desc=f"  train {e+1}/{num_epochs}", unit="batch", leave=False)
        for data, labels in train_bar:
            # CrossEntropyLoss espera índices de clase (long), no floats.
            data, labels = data.to(device, dtype=torch.float), labels.to(device, dtype=torch.long)

            optimizer.zero_grad()
            target = model(data)
            loss = criterion(target, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        valid_loss = 0.0
        val_preds_epoch, val_labels_epoch = [], []
        model.eval()
        val_bar = tqdm(validation, desc=f"  val   {e+1}/{num_epochs}", unit="batch", leave=False)
        with torch.no_grad():
            for data, labels in val_bar:
                data, labels = data.to(device, dtype=torch.float), labels.to(device, dtype=torch.long)

                target = model(data)
                loss = criterion(target, labels)
                valid_loss += loss.item()
                val_bar.set_postfix(loss=f"{loss.item():.4f}")

                val_preds_epoch.extend(torch.argmax(target, dim=1).cpu().numpy())
                val_labels_epoch.extend(labels.cpu().numpy())

        avg_train_loss = train_loss / len(train)
        avg_valid_loss = valid_loss / len(validation)
        avg_valid_f1 = f1_score(val_labels_epoch, val_preds_epoch, average="macro")
        train_loss_history.append(avg_train_loss)
        val_loss_history.append(avg_valid_loss)
        val_f1_history.append(avg_valid_f1)
        epoch_bar.set_postfix(
            train_loss=f"{avg_train_loss:.4f}", val_loss=f"{avg_valid_loss:.4f}", val_f1=f"{avg_valid_f1:.4f}",
        )

        if wandb_run is not None:
            wandb_run.log(
                {
                    "train/loss": avg_train_loss,
                    "val/loss": avg_valid_loss,
                    "val/f1_macro": avg_valid_f1,
                },
                step=e + 1,
            )

        print(
            f"Epoch {e+1} \t\t Training Loss: {avg_train_loss} \t\t "
            f"Validation Loss: {avg_valid_loss} \t\t Validation F1-macro: {avg_valid_f1:.6f}"
        )

        torch.save(model.state_dict(), LAST_MODEL_PATH)

        if avg_valid_f1 > best_val_f1:
            print(f"Validation F1-macro increased({best_val_f1:.6f}--->{avg_valid_f1:.6f}) \t Saving The Model")
            best_val_f1 = avg_valid_f1
            best_epoch = e + 1
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            counter = 0
        else:
            counter += 1
            print(f"EarlyStopping counter: {counter} out of {patience}")

        if counter >= patience:
            print(f"Early stopping triggered en la época {e+1}. Mejor época: {best_epoch}.")
            break

        if sheduler is not None:
            sheduler.step()

    print(f"\nMejor época: {best_epoch} (val F1-macro {best_val_f1:.6f}) -> {BEST_MODEL_PATH}")
    print(f"Última época: {len(train_loss_history)} -> {LAST_MODEL_PATH}")
    return model

In [ ]:
class FullModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.linear1  = nn.Linear(2048, 512)
        self.bn1      = nn.BatchNorm1d(512)
        self.act1     = nn.ReLU(inplace=True)
        self.dropout1 = nn.Dropout(p=DROPOUT)

        # 2 logits (benigno/maligno) para CrossEntropyLoss, en vez de 1 logit
        # + sigmoid (BCEWithLogitsLoss).
        self.linear2  = nn.Linear(512, 2)

    def forward(self, x):

        x = self.backbone(x)
        x = x.view(x.size(0), -1) 
         
        x = self.linear1(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.dropout1(x)
        x = self.linear2(x)
        return x  # logits [batch, 2] crudos; CrossEntropyLoss aplica softmax internamente


# Re-sembrar aquí hace que los pesos iniciales de la cabeza (linear1/bn1/
# linear2) no dependan de cuántas celdas se hayan ejecutado antes (Jupyter
# permite ejecutarlas en cualquier orden, y el RNG global es estado
# compartido).
seed_everything()  # pesos iniciales de la cabeza reproducibles
model = FullModel(backbone).to(device)

# linear1/linear2 se reinicializan con una distribución normal en vez del
# kaiming_uniform por defecto de nn.Linear. bn1 se deja en su default
# (gamma=1, beta=0); el backbone ya trae pesos RadImageNet y no se toca aquí.
for m in (model.linear1, model.linear2):
    nn.init.normal_(m.weight, mean=0.0, std=0.01)
    nn.init.zeros_(m.bias)

In [ ]:
criterion = nn.CrossEntropyLoss()

# Grupos de parámetros dinámicos: recoge lo que haya quedado con
# requires_grad=True en el backbone (cualquiera que sea UNFREEZE_IDX) en vez
# de apuntar a un índice fijo (backbone[7]) — así esta celda no cambia entre
# las 5 profundidades del bloque, solo la celda de freeze.
backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
head_params = (
    list(model.linear1.parameters())
    + list(model.bn1.parameters())
    + list(model.linear2.parameters())
)

param_groups = [{"params": head_params, "lr": LR, "name": "head"}]
if backbone_params:
    param_groups.append({"params": backbone_params, "lr": LR_BACKBONE, "name": "backbone"})

optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
sheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-7)

trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Optimizer: AdamW(lr_head={LR}, lr_backbone={LR_BACKBONE if backbone_params else None}, weight_decay={WEIGHT_DECAY})")
for g in optimizer.param_groups:
    n = sum(p.numel() for p in g["params"])
    print(f"  grupo {g['name']:9} lr={g['lr']:<8} params={n:,}")
print(f"Parámetros entrenables: {sum(p.numel() for p in trainable_params):,}")

In [ ]:
model = train_model(model, criterion, optimizer, num_epochs=NUM_EPOCHS,
                    sheduler=sheduler, patience=PATIENCE)

In [ ]:
import matplotlib.pyplot as plt

best_epoch = int(np.argmax(val_f1_history)) + 1

for epoch, (t_loss, v_loss, v_f1) in enumerate(zip(train_loss_history, val_loss_history, val_f1_history), start=1):
    marker = "  <-- best (max val F1-macro)" if epoch == best_epoch else ""
    print(
        f"Epoch {epoch:3d} \t Training Loss: {t_loss:.6f} \t Validation Loss: {v_loss:.6f} "
        f"\t Validation F1-macro: {v_f1:.6f}{marker}"
    )

loss_history_path = RUN_DIR / "loss_history.csv"
pd.DataFrame({
    "epoch": range(1, len(train_loss_history) + 1),
    "train_loss": train_loss_history,
    "val_loss": val_loss_history,
    "val_f1_macro": val_f1_history,
}).to_csv(loss_history_path, index=False)
print(f"\nHistorial guardado en: {loss_history_path}")

fig, ax = plt.subplots(figsize=(8, 5))
epochs_range = range(1, len(train_loss_history) + 1)
ax.plot(epochs_range, train_loss_history, label="Training loss", color="#1f77b4")
ax.plot(epochs_range, val_loss_history, label="Validation loss", color="#d62728")
ax.axvline(best_epoch, linestyle="--", color="gray", lw=1, label=f"Best epoch (F1-macro, {best_epoch})")
ax.set_xlim(left=0)
ax.set_ylim(0, max(max(train_loss_history), max(val_loss_history)) * 1.05)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title(f"{EXP_ID} — Training vs Validation loss (layer4, mlp head, CE, lr_head={LR}/lr_bb={LR_BACKBONE})")
ax.legend()
fig.tight_layout()

loss_curve_path = PLOTS_DIR / "loss_curve.png"
fig.savefig(loss_curve_path, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {loss_curve_path}")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

all_labels, all_preds, all_probs = [], [], []
with torch.no_grad():
    for data, labels in tqdm(test, desc="Test", unit="batch"):
        data = data.to(device, dtype=torch.float)
        logits = model(data)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        # probs[:, 1] = P(clase 1), score que roc_auc_score espera en binario.
        all_probs.extend(probs[:, 1].cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_auc = roc_auc_score(all_labels, all_probs)

idx_to_label = {v: k for k, v in train_processed.label_to_idx.items()}
print(f"label_to_idx: {train_processed.label_to_idx}  (clase positiva para AUC = {idx_to_label[1]!r})")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test ROC AUC:  {test_auc:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=[idx_to_label[0], idx_to_label[1]]))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, roc_curve

test_precision = precision_score(all_labels, all_preds, pos_label=1)
test_recall = recall_score(all_labels, all_preds, pos_label=1)
test_f1 = f1_score(all_labels, all_preds, pos_label=1)
cm = confusion_matrix(all_labels, all_preds)

class_names = [idx_to_label[0], idx_to_label[1]]

fig_cm, ax_cm = plt.subplots(figsize=(5, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax_cm, cmap="Blues", colorbar=False, values_format="d")
ax_cm.set_title(f"{EXP_ID} — Matriz de confusión (test)")
fig_cm.tight_layout()

cm_path = PLOTS_DIR / "test_confusion_matrix.png"
fig_cm.savefig(cm_path, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {cm_path}")
plt.show()

metrics = {
    "Accuracy": test_acc,
    "ROC AUC": test_auc,
    "Precision": test_precision,
    "Recall": test_recall,
    "F1-score": test_f1,
}
for name, value in metrics.items():
    print(f"{name:10s}: {value:.4f}")

if wandb_run is not None:
    wandb_run.log({f"test/{k.lower().replace(' ', '_')}": v for k, v in metrics.items()})
    wandb_run.summary["best_epoch"] = best_epoch
    wandb_run.summary["best_val_loss"] = min(val_loss_history)
    wandb_run.summary["best_val_f1_macro"] = max(val_f1_history)

# ROC AUC es el area bajo la curva ROC: se grafica la curva real, no un punto suelto.
fpr, tpr, _ = roc_curve(all_labels, all_probs)

fig_roc, ax_roc = plt.subplots(figsize=(5, 5))
ax_roc.plot(fpr, tpr, color="#1f77b4", lw=2, label=f"ROC (AUC = {test_auc:.3f})")
ax_roc.fill_between(fpr, tpr, alpha=0.1, color="#1f77b4")
ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray", lw=1, label="Azar (AUC = 0.5)")
ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1.02)
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.set_title(f"{EXP_ID} — Curva ROC (test)")
ax_roc.legend(loc="lower right")
fig_roc.tight_layout()

roc_path = PLOTS_DIR / "test_roc_curve.png"
fig_roc.savefig(roc_path, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {roc_path}")
plt.show()

# Accuracy/Precision/Recall/F1 son escalares puntuales en el mismo rango [0, 1]: se
# comparan mejor juntos en un solo grafico de barras que en 4 graficos de un punto.
bar_metrics = {k: v for k, v in metrics.items() if k != "ROC AUC"}

fig_bar, ax_bar = plt.subplots(figsize=(6, 4.5))
bars = ax_bar.bar(bar_metrics.keys(), bar_metrics.values(), color="#1f77b4")
ax_bar.set_ylim(0, 1)
ax_bar.set_ylabel("Score")
ax_bar.set_title(f"{EXP_ID} — Métricas de test")
ax_bar.grid(True, axis="y", linestyle="--", alpha=0.4)
ax_bar.bar_label(bars, fmt="%.3f", padding=3)
fig_bar.tight_layout()

bar_path = PLOTS_DIR / "test_metrics_bar.png"
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {bar_path}")
plt.show()

if wandb_run is not None:
    wandb_run.log({
        "test/confusion_matrix": wandb.Image(str(cm_path)),
        "test/roc_curve": wandb.Image(str(roc_path)),
        "test/metrics_bar": wandb.Image(str(bar_path)),
    })
    wandb_run.finish()


In [ ]:
summary = {
    "experiment": EXP_ID,
    "block": BLOCK,
    "run_name": RUN_NAME,
    "unfrozen": UNFROZEN_DESC,
    "unfreeze_idx": UNFREEZE_IDX,
    "seed": SEED,
    "hyperparams": {
        "head": HEAD,
        "dropout": DROPOUT if HEAD == "mlp" else None,
        "lr": LR,
        "lr_backbone": LR_BACKBONE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "patience": PATIENCE,
        "image_size": IMAGE_SIZE,
        "optimizer": "AdamW",
        "loss": "CrossEntropyLoss",
        "scheduler": "CosineAnnealingLR",
        "augmentation": f"RandomRotation({ROTATION_DEGREES}) + RandomHorizontalFlip(p=0.5)",
        "seed": SEED,
        "best_checkpoint_metric": "val_f1_macro",
    },
    "epochs_run": len(train_loss_history),
    "best_epoch": best_epoch,
    "best_val_loss": float(min(val_loss_history)),
    "best_val_f1_macro": float(max(val_f1_history)),
    "final_train_loss": float(train_loss_history[-1]),
    "final_val_loss": float(val_loss_history[-1]),
    "test": {
        "accuracy": float(test_acc),
        "auc": float(test_auc),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
        "confusion_matrix": cm.tolist(),
        "class_names": class_names,
    },
    "checkpoints": {
        "best": str(BEST_MODEL_PATH),
        "last": str(LAST_MODEL_PATH),
    },
}

metrics_path = RUN_DIR / "metrics.json"
metrics_path.write_text(json.dumps(summary, indent=2))
print(f"Resumen guardado en: {metrics_path}")
print(json.dumps(summary["test"], indent=2))